# 🎙️ 長時間会議 自動文字起こし・議事録作成システム

**version: 2.0 / 2026-04-19**
**最終更新日: 2026-08-21 10:02**

## ⚠️ 実行前に必ず確認！
> **PCのスリープをオフにしてください。**
> 3時間以上の録音では処理に長時間かかります。スリープすると Colab が切断され最初からやり直しになります。
>
> **Windows：** 設定 → システム → 電源とスリープ → スリープを「なし」に変更
> **Mac：** システム設定 → バッテリー → スリープしない に変更
>
> 処理が完了したら元の設定に戻してください。

※サブPCとかある人はそれ使ってもいいかもしれません。作業が終わるまで。

---

## 使い方
0. **ランタイムをGPUに設定する**（「ランタイム」→「ランタイムのタイプを変更」→ T4 GPU）
1. `MeetingTranscript/01_input` フォルダに音声ファイルを入れる
2. スプレッドシートの「本日の参加者」を更新する
3. **Step 1のセルを実行**（初回のみ・セッションが自動で再起動されます）
4. **Step 2のセルを実行**（認証ポップアップが出たら許可する）
5. Step 2完了後 → **「▶すべてのセルを実行▼」→「現在のセルと以下のすべてのセルを実行」** で待つだけ ☕

## 💡 話者分離の精度を上げるコツ
スプレッドシートの「**プロフィール**」欄に、その人の**話し方の特徴**や**経緯や、状況**を書いておくと精度が上がります。

例：
| 名前 | 役割 | プロフィール |
|---|---|---|
| 山田 太郎 | 税理士 | 〇〇税理士法人。数字や法律用語をよく使う。今回の会話で主に、説明をしてる人です。 |
| 田中 花子 | 社長 | 株式会社△△代表。山田氏に相談に。相手の言葉を自分の言葉に解釈できる言葉に変換していう傾向がある。「なるほど～ですね」などの相槌が多い |

---

## Step 1: 環境セットアップ
必要なライブラリをインストールします（初回のみ数分かかります）

In [ ]:
# ライブラリのインストール
# ※ 初回実行時は数分かかります
import time
start = time.time()

# uv（高速パッケージマネージャ）を使って一括インストール
!pip install -q uv
!uv pip install whisperx gspread google-auth google-generativeai pydub "numba>=0.61"

# Colab の古い numpy ファイルが残って混在するのを防ぐためクリーン再インストール
!pip install -q --force-reinstall --no-cache-dir numpy

elapsed = int(time.time() - start)
print(f'✅ ライブラリのインストール完了（{elapsed}秒）')

# 🔔 完了音で通知（ド・ミ・ソの上昇音）
from IPython.display import Javascript, display
display(Javascript("""
var ctx = new AudioContext();
[440, 550, 660].forEach((freq, i) => {
  var osc = ctx.createOscillator();
  var gain = ctx.createGain();
  osc.connect(gain); gain.connect(ctx.destination);
  osc.frequency.value = freq;
  gain.gain.setValueAtTime(0.3, ctx.currentTime + i * 0.15);
  gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + i * 0.15 + 0.3);
  osc.start(ctx.currentTime + i * 0.15);
  osc.stop(ctx.currentTime + i * 0.15 + 0.3);
});
"""))

print('🔄 ランタイムを自動で再起動します（数秒後に再接続してください）...')

# インストール後に自動でランタイムを再起動（手動再起動の代わり）　わざとクラッシュさせてます。
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

---
## Step 2: 🔑 認証・設定（ここだけ操作、あとは放置）
Google Drive・スプレッドシート・Notionの認証をまとめて行います。
ポップアップが出たら「許可」を押してください。

In [ ]:
import os
import glob
import requests
import gspread
from google.colab import drive, auth, userdata
from google.auth import default

# ========================================
# 1. APIキーの読み込み
# ========================================
GEMINI_API_KEY     = userdata.get('GEMINI_API_KEY')
NOTION_TOKEN       = userdata.get('NOTION_TOKEN')
NOTION_DATABASE_ID = userdata.get('NOTION_DATABASE_ID')

missing = [name for name, val in [
    ('GEMINI_API_KEY', GEMINI_API_KEY),
    ('NOTION_TOKEN', NOTION_TOKEN),
    ('NOTION_DATABASE_ID', NOTION_DATABASE_ID),
] if not val]
if missing:
    raise ValueError(f'❌ 以下のシークレットが未登録です: {missing}')
print('✅ APIキー読み込み完了')

# ========================================
# 2. 設定
# ========================================
BASE_DIR          = '/content/drive/MyDrive/MeetingTranscript'
INPUT_AUDIO_DIR   = f'{BASE_DIR}/01_input'
DONE_AUDIO_DIR    = f'{BASE_DIR}/02_processed'
TEXT_OUTPUT_DIR   = f'{BASE_DIR}/03_output'
SPREADSHEET_NAME  = 'MeetingTranscript'
WORKSHEET_NAME    = '本日の参加者'

# --- Gemini モデル選択 ---
# RPD制限に引っかかった場合は別のモデルに切り替えてください
# モデル名          RPM  RPD
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'  # 15  500  ← おすすめ
# GEMINI_MODEL = 'gemini-2.5-flash-lite'         # 10   20
# GEMINI_MODEL = 'gemini-2.5-flash'              #  5   20
# GEMINI_MODEL = 'gemini-3-flash-preview'        #  5   20

CHUNK_SIZE        = 10000  # Gemini への分割単位（文字数）

# ========================================
# 3. Google Drive マウント
# ========================================
drive.mount('/content/drive')
print('✅ Google Drive マウント完了')

# ========================================
# 4. Google 認証（Drive・Sheets 共通）
# ========================================

# 🔔 認証ポップアップが出ます！（ポーン音で通知）
from IPython.display import Javascript, display
display(Javascript("""
var ctx = new AudioContext();
var osc = ctx.createOscillator();
var gain = ctx.createGain();
osc.connect(gain); gain.connect(ctx.destination);
osc.frequency.value = 880;
gain.gain.setValueAtTime(0.3, ctx.currentTime);
gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + 0.6);
osc.start(ctx.currentTime);
osc.stop(ctx.currentTime + 0.6);
"""))
print('🔔 認証ポップアップが出ます！「許可」を押してください')

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

spreadsheet = gc.open(SPREADSHEET_NAME)
worksheet   = spreadsheet.worksheet(WORKSHEET_NAME)
records     = worksheet.get_all_records()
if not records:
    raise ValueError('❌ スプレッドシートに参加者情報がありません。更新してください。')

print(f'✅ 参加者情報を取得しました（{len(records)}名）')
for row in records:
    print(f"   - {row['名前']}（{row['役割']}）")

# ========================================
# 5. Notion 疎通確認
# ========================================
NOTION_HEADERS = {
    'Authorization': f'Bearer {NOTION_TOKEN}',
    'Notion-Version': '2022-06-28',
    'Content-Type': 'application/json',
}
db_id_raw = NOTION_DATABASE_ID.replace('-', '').strip()
db_id = f'{db_id_raw[0:8]}-{db_id_raw[8:12]}-{db_id_raw[12:16]}-{db_id_raw[16:20]}-{db_id_raw[20:32]}'

res = requests.get(f'https://api.notion.com/v1/databases/{db_id}', headers=NOTION_HEADERS)
if not res.ok:
    raise RuntimeError(f'❌ Notion 接続エラー {res.status_code}: {res.text}')
db_title = res.json().get('title', [{}])[0].get('plain_text', '不明')
print(f'✅ Notion 接続OK: {db_title}')

# ========================================
# 6. 音声・動画ファイルの確認（ffmpeg対応形式・大文字小文字どちらも可）
# ========================================
SUPPORTED_EXTS = [
    # 音声
    'mp3', 'm4a', 'wav', 'aac', 'flac', 'ogg',
    # 動画（ffmpegが音声を自動抽出）
    'mp4', 'mov', 'avi', 'mkv', 'webm', 'wmv',
]
# 大文字・小文字どちらも検索
audio_files = []
for ext in SUPPORTED_EXTS:
    audio_files += glob.glob(f'{INPUT_AUDIO_DIR}/*.{ext}')
    audio_files += glob.glob(f'{INPUT_AUDIO_DIR}/*.{ext.upper()}')

if not audio_files:
    raise FileNotFoundError(
        f'❌ {INPUT_AUDIO_DIR} に対応ファイルがありません\n'
        f'   対応形式（音声）: mp3 / m4a / wav / aac / flac / ogg\n'
        f'   対応形式（動画）: mp4 / mov / avi / mkv / webm / wmv'
    )
audio_path = audio_files[0]
print(f'✅ ファイル検出: {os.path.basename(audio_path)}')

print()
print('🎉 準備完了！あとは待つだけです ☕')

---
## Step 3: 音声ファイルの文字起こし（WhisperX）
GPUを使って音声を文字起こしします（話者分離はGeminiが担当します）

長時間音声でのGPUメモリ不足を防ぐため、まず無音部分で10分前後のチャンクに分割してから、
チャンクごとに文字起こしします（発言の途中で切れないよう、静かな瞬間を自動で探してカットします）。

In [ ]:
import os
from pydub import AudioSegment
from pydub.silence import detect_silence

# ========================================
# 無音検出で音声を10分前後のチャンクに分割する
# （長時間音声をそのままWhisperXに渡すとGPUメモリ不足になるため）
# ========================================

CHUNK_TARGET_MS   = 10 * 60 * 1000           # 目標チャンク長：10分
SEARCH_WINDOWS_MS = [60_000, 120_000, 240_000, 480_000]  # 無音を探す範囲（段階的に拡大）
MIN_SILENCE_LEN   = 700                       # これより短い無音は無視（息継ぎ等で誤分割しないため）
CHUNK_DIR         = '/content/chunks'


def find_cut_point(audio, target_ms, search_windows_ms, min_silence_len):
    """target_ms 付近で一番近い無音区間の中央（ミリ秒）を返す。見つからなければNone"""
    silence_thresh = audio.dBFS - 16  # 音声全体の平均音量より16dB静かな部分を無音とみなす
    for window in search_windows_ms:
        start = max(0, target_ms - window)
        end = min(len(audio), target_ms + window)
        silent_ranges = detect_silence(
            audio[start:end], min_silence_len=min_silence_len, silence_thresh=silence_thresh
        )
        if silent_ranges:
            best = min(silent_ranges, key=lambda r: abs((start + (r[0] + r[1]) // 2) - target_ms))
            return start + (best[0] + best[1]) // 2
    return None


print('🔊 音声を読み込んでいます...')
audio_seg = AudioSegment.from_file(audio_path)
total_ms = len(audio_seg)

# カットポイントを決める（見つからない場合は強制的に目標時刻でカット）
cut_points = [0]
cursor = CHUNK_TARGET_MS
while cursor < total_ms:
    cut = find_cut_point(audio_seg, cursor, SEARCH_WINDOWS_MS, MIN_SILENCE_LEN)
    if cut is None:
        cut = cursor
    cut_points.append(cut)
    cursor = cut + CHUNK_TARGET_MS
cut_points.append(total_ms)
cut_points = sorted(set(cut_points))

# チャンクを書き出す
os.makedirs(CHUNK_DIR, exist_ok=True)
chunk_paths = []
for i in range(len(cut_points) - 1):
    chunk = audio_seg[cut_points[i]:cut_points[i + 1]]
    chunk_path = f'{CHUNK_DIR}/chunk_{i:03d}.wav'
    chunk.export(chunk_path, format='wav')
    chunk_paths.append(chunk_path)

del audio_seg  # メモリ節約のため、書き出し後は不要

print(f'✅ 音声を {len(chunk_paths)} チャンクに分割しました（1チャンクあたり約{CHUNK_TARGET_MS // 60000}分）')

In [ ]:
import subprocess

# numpy が正常に動作するか確認し、壊れていたら修復する
try:
    from numpy._core.umath import _center  # numpy 2.x の動作確認
except ImportError:
    print('⚠️ numpy の状態が不正です。再インストールします...')
    subprocess.run(['pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', 'numpy>=2.1'], check=True)
    print('✅ numpy 再インストール完了。次のセルに進んでください。')
    raise SystemExit('numpy を再インストールしました。このセルをもう一度実行してください。')

import whisperx

# デバイスの設定（GPUが使える場合はcuda、使えない場合はcpu）
device = 'cuda'
compute_type = 'float16'

print('🎙️ WhisperX で文字起こし中... (しばらくかかります)')

# モデルの読み込み（largeモデルで高精度・チャンク全体で使い回す）
model = whisperx.load_model('large-v3', device, compute_type=compute_type)

# チャンクごとに文字起こしして結果を連結する
all_segments = []
for i, chunk_path in enumerate(chunk_paths):
    print(f'   チャンク {i + 1}/{len(chunk_paths)} を処理中...')
    chunk_audio = whisperx.load_audio(chunk_path)
    chunk_result = model.transcribe(chunk_audio, batch_size=4, language='ja')
    all_segments.extend(chunk_result['segments'])
    os.remove(chunk_path)  # 使い終わったチャンク音声は削除

result = {'segments': all_segments}
print(f'✅ 文字起こし完了（{len(result["segments"])} セグメント）')

In [ ]:
# 文字起こし結果をテキストに変換（話者ラベルなし）
transcript_lines = []
for segment in result['segments']:
    text = segment['text'].strip()
    if text:  # 空行はスキップ
        transcript_lines.append(text)

# テキスト全文を結合
full_transcript = '\n'.join(transcript_lines)

# 一時保存（途中でColabが落ちても安心）
os.makedirs(TEXT_OUTPUT_DIR, exist_ok=True)
raw_text_path = f'{TEXT_OUTPUT_DIR}/raw_transcript.txt'
with open(raw_text_path, 'w', encoding='utf-8') as f:
    f.write(full_transcript)

print(f'✅ 生テキストを保存: {raw_text_path}')
print(f'   総文字数: {len(full_transcript)} 文字')
print(f'   冒頭サンプル:')
print(full_transcript[:300])

---
## Step 4: Gemini による整形・話者分離（2段階処理）

- **Step 4a**：誤字補正・読みやすく整形（話者は気にしない）
- **Step 4b**：誰がどの発言をしたかを特定（きれいなテキストで精度アップ）

先に文章をきれいにしてから話者を特定することで、精度を担保します。

In [ ]:
import google.generativeai as genai

# Gemini の初期化
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel(GEMINI_MODEL)

# テキストを改行単位で chunk_size 文字以内に分割するユーティリティ
def split_text(text, chunk_size):
    chunks = []
    current_chunk = []
    current_len = 0
    for line in text.split('\n'):
        line_len = len(line) + 1  # 改行分を加算
        if current_len + line_len > chunk_size and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk = []
            current_len = 0
        current_chunk.append(line)
        current_len += line_len
    if current_chunk:
        chunks.append('\n'.join(current_chunk))
    return chunks

# ========================================
# Step 4a：誤字補正・整形（話者は気にしない）
# ========================================

chunks = split_text(full_transcript, CHUNK_SIZE)
print(f'✅ テキストを {len(chunks)} チャンクに分割しました')

corrected_chunks = []
for i, chunk in enumerate(chunks):
    print(f'🤖 [Step 4a] チャンク {i+1}/{len(chunks)} を誤字補正中...')

    prompt = f"""以下は音声認識で書き起こした会議のテキストです。
専門用語の明らかな誤認識・誤字を補正し、読みやすく整えてください。

【ルール】
- 内容と長さを変えずに、文章として少し整える。入力の口調と同じようにすること
- 明らかな誤字・誤変換のみ修正する
- 意味が変わるような要約・削除はしない（全文を保持）
- MarkdownやHTMLは使わない。太字（**）や記号装飾は禁止
- 話者名の付与は不要（このステップでは誰が話しているかは気にしない）

【文字起こし】
{chunk}
"""

    response = gemini.generate_content(prompt)
    corrected_chunks.append(response.text)

# 全チャンクを結合
corrected_text = '\n\n'.join(corrected_chunks)
print(f'✅ Step 4a 完了（総文字数: {len(corrected_text)} 文字）')
print(f'   冒頭サンプル:')
print(corrected_text[:300])

In [ ]:
# ========================================
# Step 4b：話者分離（きれいなテキストで精度アップ）
# ========================================

# 参加者プロフィールをテキスト化
profile_text = '\n'.join([
    f"- {row['名前']}（{row['役割']}）: {row['プロフィール']}"
    for row in records
])

chunks_4b = split_text(corrected_text, CHUNK_SIZE)
print(f'✅ テキストを {len(chunks_4b)} チャンクに分割しました')

speaker_chunks = []
for i, chunk in enumerate(chunks_4b):
    print(f'🤖 [Step 4b] チャンク {i+1}/{len(chunks_4b)} の話者を特定中...')

    prompt = f"""以下は会議の文字起こしテキストです。
参加者情報を参考に、各発言が誰の発言かを特定し、「名前：発言内容」の形式に変換してください。

【参加者情報】
{profile_text}

【ルール】
- 出力形式は必ず「名前：発言内容」（例: 田中：〜〜〜）
- 話者の特定だけに集中する。発言内容は入力テキストをそのまま使う（変えない）
- 役職・肩書き・補足は名前に付けない
- MarkdownやHTMLは使わない

【文字起こし】
{chunk}
"""

    response = gemini.generate_content(prompt)
    speaker_chunks.append(response.text)

# 全チャンクを結合
final_text = '\n\n'.join(speaker_chunks)
print(f'✅ Step 4b 完了（総文字数: {len(final_text)} 文字）')
print(f'   冒頭サンプル:')
print(final_text[:300])

---
## Step 5: Notion に議事録ページを作成

In [ ]:
from datetime import date

def notion_post(url, body):
    """Notion API に POST リクエストを送る（ページ作成用）"""
    res = requests.post(url, headers=NOTION_HEADERS, json=body)
    if not res.ok:
        raise RuntimeError(f'Notion API エラー {res.status_code}: {res.text}')
    return res.json()

def notion_patch(url, body):
    """Notion API に PATCH リクエストを送る（ブロック追記用）"""
    res = requests.patch(url, headers=NOTION_HEADERS, json=body)
    if not res.ok:
        raise RuntimeError(f'Notion API エラー {res.status_code}: {res.text}')
    return res.json()

# 本日の日付とページタイトルを生成
today = date.today().strftime('%Y-%m-%d')
page_title = f'議事録_{today}'

# 参加者リストを文字列化
participants = '、'.join([
    f"{row['名前']}（{row['役割']}）" for row in records
])

# Notion のブロック形式に変換
def text_to_blocks(text):
    """テキストを Notion の段落ブロックのリストに変換する"""
    blocks = []
    for line in text.split('\n'):
        blocks.append({
            'object': 'block',
            'type': 'paragraph',
            'paragraph': {
                'rich_text': [{'type': 'text', 'text': {'content': line}}]
            }
        })
    return blocks

print(f'📝 Notion にページを作成中: {page_title}')

# ページ作成（POST）
new_page = notion_post(
    'https://api.notion.com/v1/pages',
    {
        'parent': {'database_id': db_id},
        'properties': {
            '名前': {
                'title': [{'text': {'content': page_title}}]
            }
        }
    }
)

page_id = new_page['id']
page_url = new_page['url']
print(f'✅ ページ作成完了: {page_url}')

# 本文の冒頭に日付・参加者情報を追加（PATCH）
header_blocks = [
    {
        'object': 'block',
        'type': 'paragraph',
        'paragraph': {'rich_text': [{'type': 'text', 'text': {'content': f'日付: {today}'}}]}
    },
    {
        'object': 'block',
        'type': 'paragraph',
        'paragraph': {'rich_text': [{'type': 'text', 'text': {'content': f'参加者: {participants}'}}]}
    },
    {
        'object': 'block',
        'type': 'divider',
        'divider': {}
    }
]
notion_patch(f'https://api.notion.com/v1/blocks/{page_id}/children', {'children': header_blocks})

# 本文を100ブロックずつ分けて追加（PATCH・Notion API の制限）
all_blocks = text_to_blocks(final_text)
for i in range(0, len(all_blocks), 100):
    batch = all_blocks[i:i+100]
    notion_patch(f'https://api.notion.com/v1/blocks/{page_id}/children', {'children': batch})
    print(f'   ブロック追加中... {min(i+100, len(all_blocks))}/{len(all_blocks)}')

print(f'✅ Notion ページの作成完了！')
print(f'   🔗 {page_url}')

---
## Step 6: 後片付け

In [ ]:
import shutil
from datetime import datetime

# 処理済み音声ファイルを移動
os.makedirs(DONE_AUDIO_DIR, exist_ok=True)
done_path = os.path.join(DONE_AUDIO_DIR, os.path.basename(audio_path))
shutil.move(audio_path, done_path)
print(f'✅ 音声ファイルを移動: {done_path}')

# 最終テキストを保存
final_text_path = f'{TEXT_OUTPUT_DIR}/minutes_{today}.md'
with open(final_text_path, 'w', encoding='utf-8') as f:
    f.write(f'# {page_title}\n\n')
    f.write(f'参加者: {participants}\n\n')
    f.write('---\n\n')
    f.write(final_text)
print(f'✅ 議事録を保存: {final_text_path}')

# 処理ログをスプレッドシートに記録
try:
    log_sheet = spreadsheet.worksheet('処理ログ')
except gspread.WorksheetNotFound:
    log_sheet = spreadsheet.add_worksheet(title='処理ログ', rows=1000, cols=5)
    log_sheet.append_row(['日時', 'ファイル名', '参加者', 'Notionページ'])

log_sheet.append_row([
    datetime.now().strftime('%Y-%m-%d %H:%M'),
    os.path.basename(done_path),
    participants,
    page_url
])
print('✅ 処理ログを記録しました')

print()
print('=' * 50)
print('🎉 すべての処理が完了しました！')
print(f'   Notion ページ: {page_url}')
print('=' * 50)